# Time-Adjusted Cluster Load Allocation with Error Correction in Sparsely Metered Distribution Networks

This notebook implements the complete research pipeline for sparsely sampled distribution system state estimation, Cluster Load Allocation (CLA), and transient-assisted error correction.

In [ ]:
# Automate Wine installation if missing (required for Windows ATP-EMTP binaries on Linux research runtime)
import subprocess
try:
    subprocess.run(["wine", "--version"], check=True, capture_output=True)
    print("Wine is already installed on the research runtime.")
except Exception:
    print("Wine is missing. Installing Wine and i386 multiarch support...")
    subprocess.run("sudo dpkg --add-architecture i386 && sudo apt-get update && sudo apt-get install -y wine wine32:i386", shell=True)
    print("Wine successfully installed.")

import os
import sys
from pathlib import Path

# Kaggle / Google Colab Environment Setup  
REPO_URL = "https://github.com/mhizterpaul/dsse.git"
PYATP_URL = "https://github.com/pdb5627/pyATP.git"

if not os.path.exists("src"):
    target_dir = None
    if os.path.exists("/content"):  # Google Colab
        target_dir = "/content/dsse"
    elif os.path.exists("/kaggle/working"):  # Kaggle
        target_dir = "/kaggle/working/dsse"
    
    if target_dir:
        if not os.path.exists(target_dir):
            subprocess.run(["git", "clone", REPO_URL, target_dir], check=True)
        os.chdir(target_dir)

cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.insert(0, cwd)

if os.path.exists("/kaggle/working") or os.path.exists("/content"):
    pyatp_dir = "/kaggle/working/pyATP" if os.path.exists("/kaggle/working") else "/content/pyATP"
    if not os.path.exists(pyatp_dir):
        subprocess.run(["git", "clone", PYATP_URL, pyatp_dir], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", pyatp_dir, "--no-deps"], check=True)
    print(f"ATP utilities package ready: {pyatp_dir}")

print(f"Current working directory: {os.getcwd()}")
 
!pip install numpy scipy pywavelets pandas xarray "OpenDSSDirect.py[extras]" matplotlib altdss dss-python

import numpy as np
import scipy
import pywt
import pandas as pd
import matplotlib.pyplot as plt
print("Environment initialized successfully.")


## Stage 1: Sparsely Metered Distribution Datasets & OpenDSS Network Steady State
This section orchestrates dataset generation, displays Dataset 1 (Cluster Load Allocation energy estimation), presents the native OpenDSS network circuit graphic, and visualizes the ATP transient waveforms for 8 load circuit equipment types under OpenDSS steady state parameters.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
from src.simulation.dataset import generate_experiments_dataset

print("Stage 1: Checking dataset files...")
if not Path("src/simulation/dataset_1.csv").exists():
    dataset_1, dataset_2, dataset_3, dataset_4 = generate_experiments_dataset(write_to_disk=True)
else:
    dataset_1 = pd.read_csv("src/simulation/dataset_1.csv")
    dataset_2 = pd.read_csv("src/simulation/dataset_2.csv")
    dataset_3 = pd.read_csv("src/simulation/dataset_3.csv")
    dataset_4 = pd.read_csv("src/simulation/dataset_4.csv")


In [ ]:
import matplotlib.pyplot as plt
from src.visualization.dss_circuit_plotter import plot_opendss_circuit

print("Generating Native OpenDSS Network Circuit Plot...")
fig_circuit = plot_opendss_circuit(
    dss=runner.dss,
    use_baseline_transformers=True,
    quantity="Power",
    dots=True,
    labels=True,
    subs=True,
    save_path="src/visualization/dss_circuit_plot.png"
)
if fig_circuit:
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from src.visualization.transient_waveform_plotter import simulate_and_plot_equipment_group

display(HTML("<h3>Load Circuit Switch Transients — Equipment Group 1 (AC Motor, DC Motor Inverter, Microwave, Induction Plate)</h3>"))
fig1 = simulate_and_plot_equipment_group(1)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from src.visualization.transient_waveform_plotter import simulate_and_plot_equipment_group

display(HTML("<h3>Load Circuit Switch Transients — Equipment Group 2 (Compressor, Audio Amplifier, UPS, Industrial Fan)</h3>"))
fig2 = simulate_and_plot_equipment_group(2)
plt.show()


## Stage 2: Statistical Validation & Cluster Load Allocation Error Analysis
This section evaluates Dataset 1 energy estimation errors (Baseline CLA vs Time-Adjusted CLA), followed by Q1, Q2, and Q3 statistical correlation testing across Datasets 2, 3, and 4, and computes the final transient-assisted CLA error reduction factor.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

display(HTML("<h3>Dataset 1 (Cluster Load Allocation & Energy Estimation Dataset)</h3>"))
df_1 = pd.read_csv("src/simulation/dataset_1.csv")
display(df_1.head(20))

print("Dataset 1 Correlation & Estimation Error Testing — Baseline vs Time-Adjusted CLA Error")
unmetered_df = df_1[(df_1["consumer_type"] != "latent") & df_1["cla_estimates"].notna() & (df_1["cla_estimates"] != "")]
gt_consumer_energy = pd.to_numeric(unmetered_df["gt_consumed_energy_kwh"], errors="coerce").values
est_baseline = pd.to_numeric(unmetered_df["cla_estimates"], errors="coerce").values
est_time_adj = pd.to_numeric(unmetered_df["time_adjusted_cla_estimates"], errors="coerce").values

aggregate_gt_consumer_energy_kwh = float(np.sum(gt_consumer_energy))
aggregate_cla_estimates = float(np.sum(est_baseline))
aggregate_time_adjusted_cla_estimates = float(np.sum(est_time_adj))

baseline_cla_error_pct = float(np.abs(aggregate_gt_consumer_energy_kwh - aggregate_cla_estimates) / (aggregate_gt_consumer_energy_kwh + 1e-6)) * 100.0
time_adjusted_cla_error_pct = float(np.abs(aggregate_gt_consumer_energy_kwh - aggregate_time_adjusted_cla_estimates) / (aggregate_gt_consumer_energy_kwh + 1e-6)) * 100.0

# Mean estimation errors across individual consumer units
baseline_mean_abs_error = float(np.mean(np.abs(gt_consumer_energy - est_baseline)))
time_adjusted_mean_abs_error = float(np.mean(np.abs(gt_consumer_energy - est_time_adj)))
baseline_mean_squared_error = float(np.mean((gt_consumer_energy - est_baseline) ** 2))
time_adjusted_mean_squared_error = float(np.mean((gt_consumer_energy - est_time_adj) ** 2))
baseline_mean_pct_error = float(np.mean(np.abs(gt_consumer_energy - est_baseline) / (gt_consumer_energy + 1e-6))) * 100.0
time_adjusted_mean_pct_error = float(np.mean(np.abs(gt_consumer_energy - est_time_adj) / (gt_consumer_energy + 1e-6))) * 100.0

print(f"  Baseline Cluster Load Allocation (CLA) Aggregate Error:      {baseline_cla_error_pct:.2f}%")
print(f"  Time-Adjusted Cluster Load Allocation (CLA) Aggregate Error: {time_adjusted_cla_error_pct:.2f}%")
print(f"  Baseline CLA Mean Absolute Error across estimates:          {baseline_mean_abs_error:.4f} kWh ({baseline_mean_pct_error:.2f}% MAPE)")
print(f"  Time-Adjusted CLA Mean Absolute Error across estimates:     {time_adjusted_mean_abs_error:.4f} kWh ({time_adjusted_mean_pct_error:.2f}% MAPE)")
print(f"  Baseline CLA Mean Squared Error (MSE):                      {baseline_mean_squared_error:.4f} kWh²")
print(f"  Time-Adjusted CLA Mean Squared Error (MSE):                 {time_adjusted_mean_squared_error:.4f} kWh²")


In [ ]:
import pandas as pd
from IPython.display import display, HTML
from src.statistics.q1_event_pair_analysis import run_q1_event_pair_analysis
display(HTML("<h3>Dataset 2 (Q1 Event Pair Observability Dataset — No Time Shift, Single Baseline Tx Spec)</h3>"))
df_2 = pd.read_csv("src/simulation/dataset_2.csv")
display(df_2.head(10))
display(df_2.tail(10))
print("Question 1 Statistical Testing — Event Pair Observability (Dataset 2)")
res_q1 = run_q1_event_pair_analysis()


In [ ]:
import pandas as pd
from IPython.display import display, HTML
from src.statistics.q2_time_shift_analysis import run_q2_time_shift_analysis
display(HTML("<h3>Dataset 3 (Q2 Time Shift Operation Dataset — Single Baseline Tx Spec)</h3>"))
df_3 = pd.read_csv("src/simulation/dataset_3.csv")
display(df_3.head(10))
display(df_3.tail(10))
print("Question 2 Statistical Testing — Time Shift Operation Variation (Dataset 3)")
res_q2 = run_q2_time_shift_analysis()


In [ ]:
import pandas as pd
from IPython.display import display, HTML
from src.statistics.q3_transformer_spec_analysis import run_q3_transformer_spec_analysis
display(HTML("<h3>Dataset 4 (Q3 Transformer Specification Dataset — Fixed Time Shift = 0)</h3>"))
df_4 = pd.read_csv("src/simulation/dataset_4.csv")
display(df_4.head(10))
display(df_4.tail(10))
print("Question 3 Statistical Testing — Transformer Specification Effect (Dataset 4)")
res_q3 = run_q3_transformer_spec_analysis()


### Transient-Assisted CLA Error Reduction Factor
This cell computes the overall mean dissimilarity and error reduction factor achieved by applying transient-assisted error correction to time-adjusted CLA.

In [ ]:
import numpy as np

mean_waveform_corr = float(np.mean([
    res_q1.get("mean_waveform_correlation", 0.0),
    res_q2.get("mean_waveform_correlation", 0.0),
    res_q3.get("mean_waveform_correlation", 0.0)
]))

S = mean_waveform_corr
mean_dissimilarity = float(1.0 - S)

# Transient-assisted error reduction factor directly from waveform dissimilarity D
error_reduction_factor = mean_dissimilarity
corrected_time_adj_error_pct = float(time_adjusted_cla_error_pct * (1.0 - error_reduction_factor))

print(f"Overall Average Pearson Correlation (S = mean_waveform_corr): {S:.4f}")
print(f"Overall Mean Dissimilarity (D = 1 - S):                     {mean_dissimilarity:.4f}")
print(f"Transient-Assisted CLA Error Reduction Factor:             {error_reduction_factor:.4f} ({error_reduction_factor:.2%})")
print(f"Corrected Time-Adjusted CLA Error:                         {corrected_time_adj_error_pct:.2f}%")
